# Technical Notebook: Labyrinth A* / Dijkstra with Fire

This notebook documents the internal mechanics of the project. Focusing on: (1) the graph model implicit in the grid, (2) the A* / Dijkstra implementations, (3) dynamic fire modeling, and (4) how the PySide6 GUI orchestrates scenarios and animations.


## Quick Orientation

Project entry points:
- `Main.py`: PySide6 GUI, scenario generation (random/custom), and visualization/animation.
- `core/Algorithm.py`: shortest-path algorithms including dynamic-fire A*.
- `core/MapGenerator.py`: random maze generation (recursive backtracker + loop additions).


## Repository Layout

```text
.
├── Main.py
└── core/
    ├── Algorithm.py
    └── MapGenerator.py
```


## Grid / Graph Model

The project represents the maze as a 2D grid of characters:
- `#`: wall (impassable)
- `.`: free cell
- `D`: start
- `S`: exit
- `F`: initial fire sources

The pathfinding algorithms operate on the implicit graph where each traversable cell `(x, y)` is a node and edges connect 4-neighborhood moves:


```text
(x, y) -> (x+1, y), (x-1, y), (x, y+1), (x, y-1)

Each move has uniform cost `1`.


## Dynamic Fire Model: `compute_fire_time` (Multi-source BFS)

Dynamic fire is modeled by precomputing, for each cell, the earliest time the fire reaches it. This is done via a multi-source BFS starting from every `F` cell at `t=0`.

### Key Invariant
For any cell `(x, y)`, `fire_time[y][x]` is the minimum number of steps needed for fire to reach that cell, moving through non-wall cells (`grid != '#'`). If a cell is never reached, it remains `INF`.


In [ ]:
# core/Algorithm.py: compute_fire_time
from collections import deque
import math

INF = float('inf')

def compute_fire_time(grid):
    rows = len(grid)
    cols = len(grid[0])

    fire_time = [[INF] * cols for _ in range(rows)]
    q = deque()

    # All fire sources are initially at t = 0
    for y in range(rows):
        for x in range(cols):
            if grid[y][x] == 'F':
                fire_time[y][x] = 0
                q.append((x, y))

    def in_bounds(x, y):
        return 0 <= x < cols and 0 <= y < rows

    while q:
        x, y = q.popleft()
        t = fire_time[y][x]

        for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
            nx, ny = x + dx, y + dy
            if not in_bounds(nx, ny):
                continue
            if grid[ny][nx] == '#':
                continue

            # Fire arrives at neighbor at time t+1
            if fire_time[ny][nx] > t + 1:
                fire_time[ny][nx] = t + 1
                q.append((nx, ny))

    return fire_time


### Why BFS (Not Dijkstra) Here
All fire edges have equal cost `1` per time-step. Therefore multi-source BFS yields correct earliest-arrival times with optimal complexity `O(N*M)` over the grid (subject to walls).


## Heuristics and How Dijkstra Fits

The `Algorithm.heuristic(node, end, mode)` supports:
- `manhattan`: `|x1-x2| + |y1-y2|`
- `euclidean`: `hypot(x1-x2, y1-y2)`
- `zero`: returns `0`, which makes A* behave like Dijkstra (uniform-cost search) with a priority queue.


In [ ]:
# core/Algorithm.py: Algorithm.heuristic
import math

def heuristic(node, end, mode='manhattan'):
    x1, y1 = node
    x2, y2 = end

    if mode == 'manhattan':
        return abs(x1 - x2) + abs(y1 - y2)
    if mode == 'euclidean':
        return math.hypot(x1 - x2, y1 - y2)
    if mode == 'zero':
        return 0
    return abs(x1 - x2) + abs(y1 - y2)


## Classic A* Without Fire: `a_star_no_fire`

This variant ignores fire entirely and finds any shortest-ish path according to the selected heuristic. It uses:
- `g_cost[node]`: best-known distance from `start`
- priority queue entries `(f, node)` where `f = g + h`
- `previous[node]`: parent pointers for path reconstruction

### Note on Node Counting
The class attribute `expanded_nodes` is incremented whenever a node is popped and expanded from the open set (after checking/marking visited).


In [ ]:
# core/Algorithm.py: Algorithm.a_star_no_fire (abridged reproduction)
import heapq

INF = float('inf')

def a_star_no_fire(grid, start, end, mode='manhattan'):
    rows = len(grid)
    cols = len(grid[0])

    open_heap = []
    g_cost = {start: 0}
    heapq.heappush(open_heap, (heuristic(start, end, mode), start))

    previous = {start: None}
    visited = set()

    def in_bounds(x, y):
        return 0 <= x < cols and 0 <= y < rows


    while open_heap:
        f_current, current = heapq.heappop(open_heap)
        if current in visited:
            continue
        visited.add(current)

        if current == end:
            break


        x, y = current
        current_g = g_cost[current]

        for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
            nx, ny = x + dx, y + dy
            if not in_bounds(nx, ny):
                continue
            if grid[ny][nx] == '#':
                continue

            neighbor = (nx, ny)
            tentative_g = current_g + 1

            if tentative_g < g_cost.get(neighbor, INF):
                g_cost[neighbor] = tentative_g
                previous[neighbor] = current
                f_neighbor = tentative_g + heuristic(neighbor, end, mode)
                heapq.heappush(open_heap, (f_neighbor, neighbor))

    if end not in previous:
        return []

    # Reconstruct path
    path = []
    cur = end
    while cur is not None:
        path.append(cur)
        cur = previous[cur]
    return path[::-1]


### Path Reconstruction
The algorithm stores `previous[neighbor] = current` on relaxation. Once `end` is reached (or no path exists), the path is reconstructed by backtracking from `end` to `start` and then reversing.


## Dynamic Fire A*: `a_star_with_fire`

Dynamic fire turns the problem into a time-dependent constraint satisfaction problem: the prisoner cannot occupy a cell at time `t` if `fire_time[cell] <= t`.

### Safety Rule Used
When at node `(x, y)` with current path-length (time) `t = g_cost[current]`:
- The node is invalid if `fire_time[y][x] <= t`.
- When transitioning to neighbor `(nx, ny)`, arrival time becomes `new_t = t + 1`.
- The neighbor transition is invalid if `fire_time[ny][nx] <= new_t`.


In [ ]:
# core/Algorithm.py: Algorithm.a_star_with_fire (abridged reproduction)
import heapq

INF = float('inf')

def a_star_with_fire(grid, start, end, fire_time, mode='manhattan'):
    rows = len(grid)
    cols = len(grid[0])

    sx, sy = start
    ex, ey = end

    # If fire already reaches start at t=0, no safe solution exists
    if fire_time[sy][sx] <= 0:
        return []

    open_heap = []
    g_cost = {start: 0}
    heapq.heappush(open_heap, (heuristic(start, end, mode), start))

    previous = {start: None}
    visited = set()

    def in_bounds(x, y):
        return 0 <= x < cols and 0 <= y < rows

    while open_heap:
        f_current, current = heapq.heappop(open_heap)
        if current in visited:
            continue
        visited.add(current)

        x, y = current
        t = g_cost[current]

        # Cannot be on a cell that is already burning at time t
        if fire_time[y][x] <= t:
            continue

        if current == end:
            break


        for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
            nx, ny = x + dx, y + dy
            if not in_bounds(nx, ny):
                continue
            if grid[ny][nx] == '#':
                continue

            new_t = t + 1

            # Cannot enter neighbor if fire arrives at or before new_t
            if fire_time[ny][nx] <= new_t:
                continue


            neighbor = (nx, ny)
            if new_t < g_cost.get(neighbor, INF):
                g_cost[neighbor] = new_t
                previous[neighbor] = current
                f_neighbor = new_t + heuristic(neighbor, end, mode)
                heapq.heappush(open_heap, (f_neighbor, neighbor))

    if end not in previous:
        return []

    path = []
    cur = end
    while cur is not None:
        path.append(cur)
        cur = previous[cur]
    return path[::-1]


### Important Implementation Detail: `visited` vs time
This dynamic A* implementation uses a `visited` set keyed only by spatial node `(x, y)`. In classical A*, this is safe under admissibility/consistency assumptions, but with time-dependent constraints the *state space* might require time to be part of the state.

In this project, `fire_time[y][x]` is monotonic in the sense that once a cell is unsafe at a time, it stays unsafe for later times. That often makes spatial-only pruning reasonable in practice, but theoretically the correctness depends on more conditions than the spatial-only visited set.


## Fire Modes in the GUI: Dynamic vs Static vs None

The GUI layer decides how to transform the base grid and which algorithm variant to invoke. This happens in `LabyrinthApp.prepare_grid_and_fire()` and `LabyrinthApp.compute_all_algorithms()`.


In [ ]:
# Main.py: LabyrinthApp.prepare_grid_and_fire (essential behavior)
def prepare_grid_and_fire(self):
    base = [row[:] for row in self.grid]

    if self.fire_mode == 'dynamic':
        fire_time = compute_fire_time(base)
        return base, fire_time

    if self.fire_mode == 'static':
        for y in range(self.rows):
            for x in range(self.cols):
                if base[y][x] == 'F':
                    base[y][x] = '#'
        return base, None

    # 'none': remove fire cells
    for y in range(self.rows):
        for x in range(self.cols):
            if base[y][x] == 'F':
                base[y][x] = '.'
    return base, None


### Algorithm Selection
In dynamic mode, the GUI calls `Algorithm.a_star_with_fire(..., fire_time, mode=h_mode)` where `h_mode` is `zero` for Dijkstra and `manhattan/euclidean` otherwise.
In static/none modes, the GUI calls `Algorithm.a_star_no_fire(..., mode=h_mode)`.


In [ ]:
# Main.py: LabyrinthApp.compute_all_algorithms (essential behavior)
import time

def compute_all_algorithms(self):
    grid_algo, self.fire_time = self.prepare_grid_and_fire()

    for algo in ['dijkstra', 'manhattan', 'euclidean']:
        t0 = time.perf_counter()

        h_mode = 'zero' if algo == 'dijkstra' else algo

        if self.fire_mode == 'dynamic':
            path = self.algo.a_star_with_fire(grid_algo, self.start, self.end, self.fire_time, mode=h_mode)
        else:
            path = self.algo.a_star_no_fire(grid_algo, self.start, self.end, mode=h_mode)

        elapsed_ms = (time.perf_counter() - t0) * 1000.0
        # Stores: path, expanded_nodes, success/fail, time


## Random Maze Generator: `MapGenerator`

The maze generator builds an initial spanning-tree-like labyrinth using the *Recursive Backtracker* strategy (depth-first traversal with a stack). Walls are stored as `1` and free cells as `0`.

After carving, the generator calls `add_loops(0.1)` to introduce cycles by randomly removing internal walls that are adjacent to at least two free neighbors. This improves path diversity.


In [ ]:
# core/MapGenerator.py: skeleton of generate()
import random

class MapGenerator:
    def __init__(self, width=21, height=21):
        self.width = width
        self.height = height
        self.grid = []

    def generate(self):
        # Initialize as all walls
        self.grid = [[1 for _ in range(self.width)] for _ in range(self.height)]

        start_x, start_y = 1, 1
        self.grid[start_y][start_x] = 0
        stack = [(start_x, start_y)]

        while stack:
            x, y = stack[-1]
            neighbors = []

            # Move in steps of 2, carving 1-cell-wide corridors
            directions = [(0, -2), (0, 2), (-2, 0), (2, 0)]

            for dx, dy in directions:
                nx, ny = x + dx, y + dy
                if 0 < nx < self.width - 1 and 0 < ny < self.height - 1 and self.grid[ny][nx] == 1:
                    neighbors.append((nx, ny, dx // 2, dy // 2))

            if neighbors:
                nx, ny, wx, wy = random.choice(neighbors)
                self.grid[y + wy][x + wx] = 0  # carve intermediate wall
                self.grid[ny][nx] = 0          # carve target cell
                stack.append((nx, ny))
            else:
                stack.pop()

        # Ensure entry and exit are free
        self.grid[1][1] = 0
        self.grid[self.height - 2][self.width - 2] = 0

        self.add_loops(0.1)
        return self.grid


## GUI Orchestration and Animation

The GUI runs three phases per loaded test:
1. Prepare algorithm inputs (transform grid based on fire mode; compute fire_time if dynamic).
2. Compute and store results for all three algorithm modes (Dijkstra, Manhattan A*, Euclidean A*).
3. Animate the chosen algorithm path.


### Animation Timing Model
Animation advances one step per timer tick (100 ms). The time index `t` used for dynamic fire rendering/validation corresponds to the step index in the chosen path.


In [ ]:
# Main.py: core logic in draw_next_step() (key checks)
def draw_next_step(self):
    t = self.current_time
    x, y = self.path_to_animate[t]

    burned_now = (
        self.fire_mode == 'dynamic'
        and self.fire_time is not None
        and self.fire_time[y][x] <= t
    )

    if burned_now and not self.safe_escape:
        # Draw until last safe step, stop animation
        self.timer.stop()
        return


### Safe Escape Flag
In dynamic mode, the GUI only sets `safe_escape=True` if the chosen algorithm produced a *safe* path (i.e., `a_star_with_fire` succeeded). If dynamic A* fails for the selected heuristic, the GUI still animates a geometric path computed on a grid where fire cells are removed, purely to show where it gets blocked.


## Appendix: Requirements

The repo expects:
- `pyside6`
- `jupyter`
